## Master’s Thesis – Transkribus Export

This notebook exports already digitized magazine issues
from Transkribus using ACDH tooling.

[Documentation and where I got the code from](https://github.com/acdh-oeaw/acdh-transkribus-utils)

I used the acdh-transkribus-utils Python library (MIT License) to download and export XML from the Transkribus REST API (see https://github.com/acdh-oeaw/acdh-transkribus-utils). The original MIT license and copyright notice are preserved.



Author: Sophie Hamann
Created: 2026-01-20

In [19]:
# defining 
from pathlib import Path

DATA_BASE = Path.home() / "Documents" / "MA_Universität_Wien" / "master_thesis" / "master_thesis_data"

RAW_DIR = DATA_BASE / "raw"
PROCESSED_DIR = DATA_BASE / "processed"

RAW_TRANSKRIBUS_XML = RAW_DIR / "transkribus" / "xml"
RAW_PAGE_XML = RAW_DIR / "transkribus" / "page_xml"
RAW_PDF = RAW_DIR / "pdf" / "issues"

PROCESSED_TEI = PROCESSED_DIR / "tei"
PROCESSED_GT = PROCESSED_DIR / "training_data" / "ground_truth"

In [20]:
# make sure this directory structure exists
for d in [
    RAW_TRANSKRIBUS_XML,
    RAW_PAGE_XML,
    PROCESSED_TEI,
    PROCESSED_GT,
]:
    d.mkdir(parents=True, exist_ok=True)

In [21]:
import os

from transkribus_utils.transkribus_utils import ACDHTranskribusUtils


tr_user = os.environ.get("TRANSKRIBUS_USER")
tr_pw = os.environ.get("TRANSKRIBUS_PASSWORD")

client = ACDHTranskribusUtils(user=tr_user, password=tr_pw)

# List all Collections

In [22]:
collections = client.list_collections()
for x in collections[-7:]:
    print(x["colId"], x["colName"])

2189341 Quick Text Recognition
2204941 heresies_collection
2222244 heresies_training_dataset
2257154 Digitizing_Heresies
2291396 baseline_training_data
2297059 baseline_test
2324844 digitizing hard stuff


In [23]:
# defining working collection ID
col_id = 2204941          # heresies collection
documents = client.list_docs(col_id) # get all documents in collection

https://transkribus.eu/TrpServer/rest/collections/2204941/list


In [24]:
# inspecting 2204941 heresies_collection 
type (documents)
len(documents)

# sorting documents by title
documents = sorted(documents, key=lambda d: d["title"])
[d["title"] for d in documents]

['heresies_01',
 'heresies_02',
 'heresies_03',
 'heresies_04',
 'heresies_05',
 'heresies_06',
 'heresies_07',
 'heresies_08',
 'heresies_09',
 'heresies_10',
 'heresies_11',
 'heresies_12',
 'heresies_13',
 'heresies_14',
 'heresies_15',
 'heresies_16',
 'heresies_17',
 'heresies_18',
 'heresies_19',
 'heresies_20',
 'heresies_21',
 'heresies_22',
 'heresies_23',
 'heresies_24',
 'heresies_25',
 'heresies_26',
 'heresies_27']

In [25]:
# creating a look up to find the corresponding docId for a given document title
# "Document identifiers were retrieved programmatically via the Transkribus REST API and stored in a lookup structure 
# to ensure reproducible access and avoid hard-coded identifiers." (chat gpt)
doc_map = {d["title"]: d["docId"] for d in documents}

# Define titles to exclude (not yet digitized)
excluded_titles = {f"heresies_{i:02d}" for i in [5, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]}

# Create filtered doc_map with only available documents
doc_map_filtered = {title: doc_id for title, doc_id in doc_map.items() if title not in excluded_titles}

# Get all available docIds for download
available_doc_ids = list(doc_map_filtered.values())
available_doc_ids

[11509115,
 11501518,
 11501519,
 11501559,
 11501560,
 11501521,
 11501561,
 11501522,
 11501562,
 11501563,
 11501564]

# List all documents from the given collection (heresies_collection)

In [26]:
# checking what metadata fields are available in the collection
for k in documents[-3:]:
    print(k.keys())

dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])
dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])
dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])


In [27]:
# exploratory inspection of document metadata
documents = client.list_docs(col_id)

for doc in documents:
    print(
        doc["docId"],
        doc["title"],
        doc["nrOfPages"]
    )

https://transkribus.eu/TrpServer/rest/collections/2204941/list
11501518 heresies_02 128
11501519 heresies_03 122
11501520 heresies_05 139
11501521 heresies_07 130
11501522 heresies_09 100
11501523 heresies_13 100
11501559 heresies_04 132
11501560 heresies_06 132
11501561 heresies_08 132
11501562 heresies_10 100
11501563 heresies_11 98
11501564 heresies_12 100
11501565 heresies_14 42
11501566 heresies_15 76
11501567 heresies_16 98
11501568 heresies_17 100
11501569 heresies_18 97
11501570 heresies_19 33
11501571 heresies_20 100
11501573 heresies_24 42
11501574 heresies_25 100
11501775 heresies_21 84
11501777 heresies_22 100
11501779 heresies_23 100
11501781 heresies_26 112
11501783 heresies_27 121
11509115 heresies_01 116


# Inspect one Document from the Given Collection
This part of the code inspects a single document out of the Transkribus heresies_collection.
I am looking at document metadata, page lists, page xml and transcripts.

All Python code used in this thesis for accessing, inspecting, and exporting Transkribus documents is based exclusively on the publicly available and MIT-licensed acdh-transkribus-utils library, without introducing additional custom functionality beyond the methods provided in the original repository.
[see here](https://github.com/acdh-oeaw/acdh-transkribus-utils/blob/main/transkribus_utils/transkribus_utils.py)

In [28]:
# defining document ID
doc_id_1 = 11509115         # heresies_01

In [29]:
# inspecting document metadata
doc_metadata_1 = client.get_doc_md(doc_id_1, col_id)
print(doc_metadata_1)    

{'type': 'trpDocMetadata', 'docId': 11509115, 'title': 'heresies_01', 'uploadTimestamp': 1760968484613, 'uploader': 'a12248219@unet.univie.ac.at', 'uploaderId': 447621, 'nrOfPages': 116, 'pageId': 104610900, 'url': 'https://files.transkribus.eu/Get?fileType=view&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'thumbUrl': 'https://files.transkribus.eu/Get?fileType=thumb&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'status': 0, 'fimgStoreColl': 'TrpDoc_DEA_11509115', 'origDocId': 0, 'collectionList': {'colList': [{'colId': 2204941, 'colName': 'heresies_collection', 'description': 'created by a12248219@unet.univie.ac.at', 'crowdsourcing': False, 'elearning': False, 'nrOfDocuments': 0, 'isFavorite': False}]}, 'attributes': [], 'labels': [{'labelId': 127, 'name': 'Ready for Export', 'color': '#1E90FF'}], 'pageLabels': [], 'mainColId': 2204941}


In [30]:
# inspect document overview in markdown format
overview = client.get_doc_overview_md(doc_id_1, col_id)
print(overview["trp_return"]["md"])

{'nrOfRegions': 1029, 'nrOfTranscribedRegions': 0, 'nrOfWordsInRegions': 0, 'nrOfLines': 8712, 'nrOfTranscribedLines': 8706, 'nrOfWordsInLines': 63655, 'nrOfWords': 0, 'nrOfTranscribedWords': 0, 'nrOfCharsInLines': 305727, 'nrOfTables': 0, 'nrOfNew': 0, 'nrOfInProgress': 0, 'nrOfDone': 0, 'nrOfFinal': 116, 'nrOfGT': 0, 'docId': 11509115, 'title': 'heresies_01', 'uploadTimestamp': 1760968484613, 'uploader': 'a12248219@unet.univie.ac.at', 'uploaderId': 447621, 'nrOfPages': 116, 'pageId': 104610900, 'url': 'https://files.transkribus.eu/Get?fileType=view&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'thumbUrl': 'https://files.transkribus.eu/Get?fileType=thumb&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'status': 0, 'fimgStoreColl': 'TrpDoc_DEA_11509115', 'origDocId': 0, 'collectionList': {'colList': [{'colId': 2204941, 'colName': 'heresies_collection', 'description': 'created by a12248219@unet.univie.ac.at', 'crowdsourcing': False, 'elearning': False, 'nrOfDocuments': 0, 'isFavorite': False}]}, 'attributes': [], 'label

In [31]:
#inspecting full xml
fulldoc = client.get_fulldoc_md(doc_id_1, col_id, page_id=5)
print(fulldoc["doc_xml"])

<Element trpPage at 0x1128dbbc0>


In [32]:
fulldoc = client.get_transcript(fulldoc)
print(fulldoc["transcript"][:10])

['3', 'the same power and the same intent, and in-', 'dicating that word and image can be equal', 'ingredients in politically effective art.', 'We found no solutions to the issues raised,', 'but we are finding approaches that feel fresher', 'and more satisfying. Working together toward', 'collective decisions was entirely different from', 'working alone or as part of conventional hier-', 'archies. Each of us worked on every page of this']


In [33]:
# I want to see the xml structure of a page
from lxml import etree

print(etree.tostring(fulldoc["doc_xml"], pretty_print=True).decode())

<trpPage>
  <pageId>104610902</pageId>
  <docId>11509115</docId>
  <pageNr>5</pageNr>
  <key>AUWVBEUKXASYBYSDIASPIETA</key>
  <imageId>87781844</imageId>
  <url>https://files.transkribus.eu/Get?fileType=view&amp;id=AUWVBEUKXASYBYSDIASPIETA</url>
  <thumbUrl>https://files.transkribus.eu/Get?fileType=thumb&amp;id=AUWVBEUKXASYBYSDIASPIETA</thumbUrl>
  <md5Sum>57dfca623b101345334a38ac5f2d1e4f</md5Sum>
  <fileSize>1421330</fileSize>
  <imgFileName>p005.jpg</imgFileName>
  <hideOnSites>false</hideOnSites>
  <tsList>
    <transcripts>
      <tsId>272217757</tsId>
      <parentTsId>272203585</parentTsId>
      <key>QOUUUHRRJYSTXDODJNPJTOMH</key>
      <pageId>104610902</pageId>
      <docId>11509115</docId>
      <pageNr>5</pageNr>
      <url>https://files.transkribus.eu/Get?id=QOUUUHRRJYSTXDODJNPJTOMH</url>
      <status>FINAL</status>
      <userName>a12248219@unet.univie.ac.at</userName>
      <userId>447621</userId>
      <timestamp>1768564477064</timestamp>
      <toolName>PyLaia decoding

# This is just checking whether this xml file has something in it. 
This code has been generated by Chat GPT

In [34]:
fulldoc = client.get_transcript(fulldoc)
page_xml = fulldoc["page_xml"]
ns = {"page": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}
len(page_xml.xpath(".//page:TextLine", namespaces=ns))

137

In [35]:
chars = page_xml.xpath(".//page:TextLine//page:Unicode/text()", namespaces=ns)
sum(len(c) for c in chars)

6681

# Download METS files from Collection
The METS files exported from Transkribus serve as structural containers and reference PAGE-XML files, in which the actual transcribed text is encoded within <TextLine> and <Unicode> elements.

In [18]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils

COL_ID = 2204941          # heresies collection

client.collection_to_mets(
    COL_ID,
    file_path=str(RAW_TRANSKRIBUS_XML)
)

https://transkribus.eu/TrpServer/rest/collections/2204941/list
27 to download
saving: /Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/raw/transkribus/xml/2204941/11501518_mets.xml
saving: /Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/raw/transkribus/xml/2204941/11501518_image_name.xml
1/27


KeyboardInterrupt: 

This code exports the METS XML for a single specified document from a Transkribus collection and stores it locally.

In [ ]:
COL_ID = 2204941          # heresies collection
DOC_ID = 11509115  

client = ACDHTranskribusUtils()

client.collection_to_mets(
    COL_ID,
    file_path="./foo",
    filter_by_doc_ids=[DOC_ID]
)

# Download PAGE-XML with the actual text (issue 1)
# DONE WITH CHAT GPT (cite correctly!)

In [36]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils
from lxml import etree
import os

client = ACDHTranskribusUtils()

COL_ID = 2204941
DOC_ID = 11509115  # heresies_01

In [37]:
overview = client.get_doc_overview_md(DOC_ID, COL_ID)
pages = overview["pages"]

In [39]:
# one file per page for all available documents
for title, doc_id in doc_map_filtered.items():
    # Create directory for each document
    output_dir = f"{title}_pagexml"
    os.makedirs(output_dir, exist_ok=True)
    
    # Get pages for this document
    overview = client.get_doc_overview_md(doc_id, COL_ID)
    pages = overview["pages"]
    
    # Download each page
    for p in pages:
        page_id = p["page_nr"]

        fulldoc = client.get_fulldoc_md(doc_id, COL_ID, page_id=page_id)
        fulldoc = client.get_transcript(fulldoc)

        page_xml = fulldoc["page_xml"]

        out_file = f"{output_dir}/page_{page_id}.xml"
        with open(out_file, "wb") as f:
            f.write(etree.tostring(page_xml, pretty_print=True))

In [ ]:
# one file per issue
from lxml import etree
import os

root = etree.Element("Document")

for p in pages:
    page_id = p["page_nr"]

    fulldoc = client.get_fulldoc_md(DOC_ID, COL_ID, page_id=page_id)
    fulldoc = client.get_transcript(fulldoc)

    page_xml = fulldoc["page_xml"]

    # import PAGE-XML under a single root
    root.append(page_xml)

# write one combined file
with open("heresies_01_pagexml.xml", "wb") as f:
    f.write(etree.tostring(root, pretty_print=True))